In [ ]:
import numpy as np
import glob
import plotly.graph_objects as go

LIMB_GROUPS = {
    "L-Arm":  ["leftshoulder", "leftarm",  "leftforearm",  "lefthand"],
    "R-Arm":  ["rightshoulder","rightarm", "rightforearm", "righthand"],
    "L-Leg":  ["leftupleg",   "leftleg",  "leftfoot",     "lefttoebase"],
    "R-Leg":  ["rightupleg",  "rightleg", "rightfoot",    "righttoebase"],
    "Head":   ["neck",        "head"],
    "Spine":  ["hips", "spine", "chest"],
}

LIMB_COLORS = {
    "L-Arm":  (65,  105, 225),
    "R-Arm":  (255, 140,   0),
    "L-Leg":  (46,  139,  87),
    "R-Leg":  (147, 112, 219),
    "Head":   (220,  20,  60),
    "Spine":  (120, 120, 120),
}

# Per-frame skeleton opacity — low so overlapping frames build up naturally
SKELETON_ALPHA = 0.07   # tweak: more frames → lower; fewer frames → higher
MEAN_LINE_WIDTH   = 3
FRAME_LINE_WIDTH  = 0.8


def match_limb(joint_name, keywords):
    jn = joint_name.lower().replace("mixamorig:", "").replace("_", "").replace(" ", "")
    return any(kw in jn for kw in keywords)


def rgba(rgb, a):
    return f"rgba({rgb[0]},{rgb[1]},{rgb[2]},{a})"


def proc(data_path, data2_path=None):
    motion_files = glob.glob(f"{data_path}/*.npz")
    all_motions  = []
    for f in motion_files:
        motion_data = np.load(f)
        all_motions.append(motion_data["joints_2d"][..., :2])

    all_motions = np.concatenate(all_motions, axis=0)   # (N, J, 2)
    joint_names = motion_data["joint_names"].tolist()
    bones       = motion_data["bones"].tolist()
    edges       = [[joint_names.index(b[0]), joint_names.index(b[1])] for b in bones]
    mean_motion = np.mean(all_motions, axis=0)           # (J, 2)
    N           = len(all_motions)

    # Map each edge to a limb group (for colouring)
    def edge_limb(i0, i1):
        for limb, kws in LIMB_GROUPS.items():
            if match_limb(joint_names[i0], kws) or match_limb(joint_names[i1], kws):
                return limb
        return "Spine"

    edge_limbs = [edge_limb(e[0], e[1]) for e in edges]

    # ── Build per-limb batched line data ─────────────────────────────────────
    # Each edge across all N frames: pack as NaN-separated segments into one trace
    # This keeps total trace count = n_limbs (not N×n_edges)
    limb_frame_x = {l: [] for l in LIMB_GROUPS}
    limb_frame_y = {l: [] for l in LIMB_GROUPS}

    for (i0, i1), limb in zip(edges, edge_limbs):
        xs = all_motions[:, i0, 0]   # (N,)
        x1s = all_motions[:, i1, 0]
        ys = all_motions[:, i0, 1]
        y1s = all_motions[:, i1, 1]

        # Interleave: x0, x1, NaN  for each frame → one long flat array
        seg_x = np.empty(N * 3)
        seg_y = np.empty(N * 3)
        seg_x[0::3] = xs;   seg_x[1::3] = x1s;  seg_x[2::3] = np.nan
        seg_y[0::3] = ys;   seg_y[1::3] = y1s;  seg_y[2::3] = np.nan

        limb_frame_x[limb].append(seg_x)
        limb_frame_y[limb].append(seg_y)

    fig = go.Figure()

    # ── 1. All-frame skeletons (one trace per limb) ───────────────────────────
    for limb in LIMB_GROUPS:
        if not limb_frame_x[limb]:
            continue
        rgb = LIMB_COLORS[limb]
        fig.add_trace(go.Scatter(
            x=np.concatenate(limb_frame_x[limb]),
            y=np.concatenate(limb_frame_y[limb]),
            mode="lines",
            line=dict(color=rgba(rgb, SKELETON_ALPHA), width=FRAME_LINE_WIDTH),
            hoverinfo="skip",
            showlegend=False,
            legendgroup=limb,
        ))

    # ── 2. Mean skeleton on top ───────────────────────────────────────────────
    for (i0, i1), limb in zip(edges, edge_limbs):
        rgb = LIMB_COLORS[limb]
        fig.add_trace(go.Scatter(
            x=[mean_motion[i0, 0], mean_motion[i1, 0]],
            y=[mean_motion[i0, 1], mean_motion[i1, 1]],
            mode="lines",
            line=dict(color=rgba(rgb, 1.0), width=MEAN_LINE_WIDTH),
            showlegend=False,
            legendgroup=limb,
        ))

    # Mean joint dots + legend entry per limb
    for limb, kws in LIMB_GROUPS.items():
        idxs = [i for i, jn in enumerate(joint_names) if match_limb(jn, kws)]
        if not idxs:
            continue
        rgb = LIMB_COLORS[limb]
        fig.add_trace(go.Scatter(
            x=mean_motion[idxs, 0],
            y=mean_motion[idxs, 1],
            mode="markers",
            marker=dict(size=6, color=rgba(rgb, 1.0),
                        line=dict(color="white", width=1)),
            text=[joint_names[i] for i in idxs],
            hovertemplate="%{text}<br>x=%{x:.1f}, y=%{y:.1f}<extra></extra>",
            name=limb,
            legendgroup=limb,
            showlegend=True,
        ))

    # ── 3. Boundary box ───────────────────────────────────────────────────────
    fig.add_shape(
        type="rect", x0=0, y0=0, x1=w, y1=h,
        line=dict(color="red", width=2, dash="dash"),
    )

    fig.update_layout(
        title=f"All {N} frames as skeletons (α={SKELETON_ALPHA}) + mean",
        xaxis=dict(title="x", range=[-50, w + 50], zeroline=True),
        yaxis=dict(title="y", range=[h + 50, -50], zeroline=True,
                   scaleanchor="x", scaleratio=1),
        width=1100, height=750,
        template="plotly_white",
        legend=dict(
            groupclick="togglegroup",
            title="Limb (click to toggle)",
        ),
    )
    fig.show()

data_path = "/host/data/mint/Motion_Dataset/Mixamo/single_character/{}set_motion/rdy_to_wan/5_frames/michelle"
h = 720
w = 1280
proc(data_path.format("train"))
proc(data_path.format("test"))

In [36]:
import numpy as np
import glob
import plotly.graph_objects as go

DATASETS = {
    "train": "/host/data/mint/Motion_Dataset/Mixamo/single_character/trainset_motion/rdy_to_wan/5_frames/all",
    "test":  "/host/data/mint/Motion_Dataset/Mixamo/single_character/testset_motion/rdy_to_wan/5_frames/all",
}

DATASET_COLORS = {
    "train": (220,  50,  50),
    "test":  ( 50, 100, 220),
}

LIMB_GROUPS = {
    "L-Arm":  ["leftshoulder", "leftarm",  "leftforearm",  "lefthand"],
    "R-Arm":  ["rightshoulder","rightarm", "rightforearm", "righthand"],
    "L-Leg":  ["leftupleg",   "leftleg",  "leftfoot",     "lefttoebase"],
    "R-Leg":  ["rightupleg",  "rightleg", "rightfoot",    "righttoebase"],
    "Head":   ["neck",        "head"],
    "Spine":  ["hips",        "spine",    "chest"],
}

h = 720
w = 1280
FRAME_LINE_WIDTH = 0.8
MEAN_LINE_WIDTH  = 3


def rgba(rgb, a):
    return f"rgba({rgb[0]},{rgb[1]},{rgb[2]},{a})"


def match_limb(joint_name, keywords):
    jn = joint_name.lower().replace("mixamorig:", "").replace("_", "").replace(" ", "")
    return any(kw in jn for kw in keywords)


def edge_to_limb(i0, i1, joint_names):
    for limb, kws in LIMB_GROUPS.items():
        if match_limb(joint_names[i0], kws) or match_limb(joint_names[i1], kws):
            return limb
    return "Spine"


def load_dataset(data_path):
    motion_files = glob.glob(f"{data_path}/*.npz")
    all_motions  = []
    for f in motion_files:
        motion_data = np.load(f)
        all_motions.append(motion_data["joints_2d"][..., :2])
    all_motions = np.concatenate(all_motions, axis=0)
    joint_names = motion_data["joint_names"].tolist()
    bones       = motion_data["bones"].tolist()
    edges       = [[joint_names.index(b[0]), joint_names.index(b[1])] for b in bones]
    return all_motions, joint_names, edges


def build_limb_lines(all_motions, edges, joint_names):
    N = len(all_motions)
    limb_x = {l: [] for l in LIMB_GROUPS}
    limb_y = {l: [] for l in LIMB_GROUPS}
    for i0, i1 in edges:
        limb = edge_to_limb(i0, i1, joint_names)
        seg_x = np.empty(N * 3)
        seg_x[0::3] = all_motions[:, i0, 0]
        seg_x[1::3] = all_motions[:, i1, 0]
        seg_x[2::3] = np.nan
        seg_y = np.empty(N * 3)
        seg_y[0::3] = all_motions[:, i0, 1]
        seg_y[1::3] = all_motions[:, i1, 1]
        seg_y[2::3] = np.nan
        limb_x[limb].append(seg_x)
        limb_y[limb].append(seg_y)
    return (
        {l: np.concatenate(v) for l, v in limb_x.items() if v},
        {l: np.concatenate(v) for l, v in limb_y.items() if v},
    )


def build_limb_mean(mean_motion, edges, joint_names):
    limb_x = {l: [] for l in LIMB_GROUPS}
    limb_y = {l: [] for l in LIMB_GROUPS}
    limb_joints = {l: [] for l in LIMB_GROUPS}   # joint indices per limb
    for i0, i1 in edges:
        limb = edge_to_limb(i0, i1, joint_names)
        limb_x[limb] += [mean_motion[i0, 0], mean_motion[i1, 0], np.nan]
        limb_y[limb] += [mean_motion[i0, 1], mean_motion[i1, 1], np.nan]
        for idx in (i0, i1):
            if idx not in limb_joints[limb]:
                limb_joints[limb].append(idx)
    return (
        {l: v for l, v in limb_x.items() if v},
        {l: v for l, v in limb_y.items() if v},
        {l: v for l, v in limb_joints.items() if v},
    )


# Distinct line dash per limb so train vs test are still separable
# even in greyscale printouts
LIMB_DASH = {
    "L-Arm": "solid",
    "R-Arm": "solid",
    "L-Leg": "dot",
    "R-Leg": "dot",
    "Head":  "dash",
    "Spine": "dashdot",
}


def proc(datasets, dataset_colors):
    fig   = go.Figure()
    limbs = list(LIMB_GROUPS.keys())

    for ds_name, data_path in datasets.items():
        print(f"Loading {ds_name} ...")
        all_motions, joint_names, edges = load_dataset(data_path)
        N           = len(all_motions)
        mean_motion = np.mean(all_motions, axis=0)
        rgb         = dataset_colors[ds_name]
        alpha       = float(np.clip(3 / N, 0.1, 0.08))
        print(f"  {N} frames, alpha={alpha:.3f}")

        frame_x, frame_y             = build_limb_lines(all_motions, edges, joint_names)
        mean_x,  mean_y, mean_joints = build_limb_mean(mean_motion, edges, joint_names)

        for limb in limbs:
            if limb not in frame_x:
                continue

            group = f"{ds_name} — {limb}"
            dash  = LIMB_DASH[limb]

            # ── Ghost frames ──────────────────────────────────────────────
            fig.add_trace(go.Scatter(
                x=frame_x[limb], y=frame_y[limb],
                mode="lines",
                line=dict(color=rgba(rgb, alpha), width=FRAME_LINE_WIDTH,
                          dash=dash),
                hoverinfo="skip",
                showlegend=False,
                legendgroup=group,
            ))

            # ── Mean skeleton lines ───────────────────────────────────────
            fig.add_trace(go.Scatter(
                x=mean_x[limb], y=mean_y[limb],
                mode="lines",
                line=dict(color=rgba(rgb, 1.0), width=MEAN_LINE_WIDTH,
                          dash=dash),
                hoverinfo="skip",
                showlegend=False,
                legendgroup=group,
            ))

            # ── Mean joint dots — THIS is the legend entry ────────────────
            idxs = mean_joints[limb]
            fig.add_trace(go.Scatter(
                x=mean_motion[idxs, 0],
                y=mean_motion[idxs, 1],
                mode="markers",
                marker=dict(size=6, color=rgba(rgb, 1.0),
                            line=dict(color="white", width=1)),
                text=[joint_names[i] for i in idxs],
                hovertemplate="%{text}<br>x=%{x:.1f}, y=%{y:.1f}<extra></extra>",
                name=group,           # "train — L-Arm", "test — L-Arm", etc.
                legendgroup=group,
                showlegend=True,
                legendgrouptitle_text=ds_name if limb == limbs[0] else None,
            ))

    # ── Boundary box ──────────────────────────────────────────────────────
    fig.add_shape(
        type="rect", x0=0, y0=0, x1=w, y1=h,
        line=dict(color="grey", width=1.5, dash="dash"),
    )

    fig.update_layout(
        title="Motion distribution — colour by dataset, toggle limbs freely in legend",
        xaxis=dict(title="x", range=[-50, w + 50], zeroline=True),
        yaxis=dict(title="y", range=[h + 50, -50], zeroline=True,
                   scaleanchor="x", scaleratio=1),
        width=1200, height=800,
        template="plotly_white",
        legend=dict(
            groupclick="togglegroup",   # one click hides frames + mean + dots together
            tracegroupgap=8,
            title="click to toggle",
        ),
    )
    fig.show()


proc(DATASETS, DATASET_COLORS)

Loading train ...
  1955 frames, alpha=0.080
Loading test ...
  1125 frames, alpha=0.080
